In [ ]:
%pip install python-docx langchain

In [ ]:
from docx import Document as DocxDocument
from langchain_core.documents import Document
import re

def load_docx(file_path):
    doc = DocxDocument(file_path)
    full_text = []

    for para in doc.paragraphs:
        text = para.text.strip()
        if text:
            full_text.append(text)

    return "\n".join(full_text)


def split_law_articles(text):
    # 줄 단위 기준으로 조문 찾기
    pattern = r"(?m)^(제\d+조(?:의\d+)?(?:\s*\(.*?\))?)"

    parts = re.split(pattern, text)

    docs = []

    for i in range(1, len(parts), 2):
        article_title = parts[i].strip()
        content = parts[i + 1].strip() if i + 1 < len(parts) else ""

        full_text = f"{article_title} {content}"

        # 너무 짧은 건 제거
        if len(full_text) < 20:
            continue

        docs.append(
            Document(
                page_content=full_text,
                metadata={
                    "article": article_title
                }
            )
        )

    return docs




In [ ]:
file_path = "./documents/income_tax.docx"

text = load_docx(file_path)
document_list = split_law_articles(text)


print(f"총 {len(document_list)}개 조문 생성")

# print("\n샘플:")
# print(docs[0]["content"][:500])
# print(docs[0]["metadata"])
# docs[69]

for i, doc in enumerate(document_list):
    print(f"\n===== [{i}] =====")
    print(doc.page_content)
    print(doc.metadata)

# print(docs)

In [ ]:
%pip install -q langchain-chroma

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")



In [ ]:
from langchain_chroma import Chroma
import os

if os.path.exists("./income_tax_collection"):
    vector_store = Chroma(
        collection_name="income_tax_collection",
        persist_directory="./income_tax_collection",
        embedding_function=embeddings
    )
    print("기존 vector_store 존재")
    # vector_store.add_documents(document_list)
else:
    vector_store = Chroma.from_documents(
        documents=document_list,
        embedding=embeddings,
        collection_name="income_tax_collection",
        persist_directory="./income_tax_collection"
    )

# vector_store = Chroma.from_documents(
#     documents=document_list,
#     embedding=embeddings,
#     collection_name="income_tax_collection",
#     persist_directory="./income_tax_collection"
# )


In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
query = "연봉 5천만원 직장인의 소득세는?"

In [ ]:
retriever.invoke(query)
# print(out[1])

In [ ]:
from typing_extensions import TypedDict

class AgentState(TypedDict):
    query: str
    context: list[Document]
    answer: str
    

In [ ]:
from langgraph.graph import StateGraph

graph_builder = StateGraph(AgentState)


In [ ]:
def retrieve(state: AgentState):
    query = state["query"]
    docs = retriever.invoke(state["query"])
    return {"context": docs}


In [ ]:
# Create a LangSmith API in Settings > API Keys
# Make sure API key env var is set:
# import os; os.environ["LANGSMITH_API_KEY"] = "<your-api-key>"
from langsmith import Client
from langchain_openai import ChatOpenAI

client = Client()
prompt = client.pull_prompt("rlm/rag-prompt")

llm = ChatOpenAI(model='gpt-4o')

In [ ]:
def generate(state: AgentState):
    context = state['context']
    query = state['query']
    rag_chain = prompt | llm
    response = rag_chain.invoke({'question': query,'context': context})
    return {'answer': response}

In [ ]:
graph_builder.add_node('retrieve', retrieve)
graph_builder.add_node('generate', generate)

In [ ]:
from langgraph.graph import START, END

graph_builder.add_edge(START, 'retrieve')
graph_builder.add_edge('retrieve', 'generate')
graph_builder.add_edge('generate', END)

In [ ]:
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
initial_state = {'query': query}
graph.invoke(initial_state)